# StayPredict — 02. Feature Engineering & Preprocessing Pipeline
Engineering temporal admission features and building a leakage-free Scikit-learn preprocessing pipeline.


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load cleaned dataset
DATA_PATH = Path("../data/processed/healthcare_cleaned.csv")
df = pd.read_csv(DATA_PATH)
df['Date of Admission'] = pd.to_datetime(df['Date of Admission'])

print(f"Loaded dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head(3)


Loaded dataset: 54,966 rows, 16 columns


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results,Length of Stay
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal,2
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive,6
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal,15


In [5]:
# Engineer temporal features
df['Admission Year'] = df['Date of Admission'].dt.year
df['Admission Month'] = df['Date of Admission'].dt.month
df['Admission Day'] = df['Date of Admission'].dt.day
df['Admission Day of Week'] = df['Date of Admission'].dt.dayofweek  # 0=Monday, 6=Sunday

print("--- ENGINEERED TEMPORAL FEATURES ---")
display(df[['Date of Admission', 'Admission Year', 'Admission Month', 'Admission Day', 'Admission Day of Week']].head())


--- ENGINEERED TEMPORAL FEATURES ---


,Date of Admission,Admission Year,Admission Month,Admission Day,Admission Day of Week
0,2024-01-31,2024,1,31,2
1,2019-08-20,2019,8,20,1
2,2022-09-22,2022,9,22,3
3,2020-11-18,2020,11,18,2
4,2022-09-19,2022,9,19,0


In [6]:
# Define target
TARGET = 'Length of Stay'
y = df[TARGET].copy()

# Define selected feature columns (strictly pre-admission information)
NUMERICAL_FEATURES = ['Age', 'Admission Year', 'Admission Month', 'Admission Day', 'Admission Day of Week']
CATEGORICAL_FEATURES = ['Gender', 'Blood Type', 'Medical Condition', 'Insurance Provider', 'Admission Type']

SELECTED_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
X = df[SELECTED_FEATURES].copy()

print("--- FEATURE MATRIX (X) & TARGET (y) ---")
print(f"Features (X) Shape: {X.shape[0]:,} rows, {X.shape[1]} columns")
print(f"Target (y) Shape:   {y.shape[0]:,} rows")
print(f"\nNumerical features ({len(NUMERICAL_FEATURES)}):   {NUMERICAL_FEATURES}")
print(f"Categorical features ({len(CATEGORICAL_FEATURES)}): {CATEGORICAL_FEATURES}")
display(X.head())


--- FEATURE MATRIX (X) & TARGET (y) ---
Features (X) Shape: 54,966 rows, 10 columns
Target (y) Shape:   54,966 rows

Numerical features (5):   ['Age', 'Admission Year', 'Admission Month', 'Admission Day', 'Admission Day of Week']
Categorical features (5): ['Gender', 'Blood Type', 'Medical Condition', 'Insurance Provider', 'Admission Type']


,Age,Admission Year,Admission Month,Admission Day,Admission Day of Week,Gender,Blood Type,Medical Condition,Insurance Provider,Admission Type
0,30,2024,1,31,2,Male,B-,Cancer,Blue Cross,Urgent
1,62,2019,8,20,1,Male,A+,Obesity,Medicare,Emergency
2,76,2022,9,22,3,Female,A-,Obesity,Aetna,Emergency
3,28,2020,11,18,2,Female,O+,Diabetes,Medicare,Elective
4,43,2022,9,19,0,Female,AB+,Cancer,Aetna,Urgent


# Phase 6 & 7 — Train/Test Split and Preprocessing Pipeline
Splitting data into 80% train and 20% test, followed by fitting an isolated `ColumnTransformer` to prevent preprocessing leakage.


In [7]:
from sklearn.model_selection import train_test_split

# Split into 80% Train, 20% Test (Reproducible seed)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("--- TRAIN / TEST SPLIT SUMMARY ---")
print(f"Total Samples: {len(X):,}")
print(f"X_train shape: {X_train.shape[0]:,} rows, {X_train.shape[1]} columns")
print(f"X_test shape:  {X_test.shape[0]:,} rows, {X_test.shape[1]} columns")
print(f"y_train shape: {y_train.shape[0]:,} rows")
print(f"y_test shape:  {y_test.shape[0]:,} rows")


--- TRAIN / TEST SPLIT SUMMARY ---
Total Samples: 54,966
X_train shape: 43,972 rows, 10 columns
X_test shape:  10,994 rows, 10 columns
y_train shape: 43,972 rows
y_test shape:  10,994 rows


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Numerical Pipeline: Median Imputation + Standard Scaling (zero mean, unit variance)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline: Frequent Imputation + One-Hot Encoding
# handle_unknown='ignore' ensures future unseen categories during inference won't crash the pipeline
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Master ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, NUMERICAL_FEATURES),
        ('cat', cat_pipeline, CATEGORICAL_FEATURES)
    ],
    remainder='drop'  # Drop any unlisted columns for security
)

print("--- PREPROCESSING PIPELINE CONSTRUCTED ---")
print(preprocessor)


--- PREPROCESSING PIPELINE CONSTRUCTED ---
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Admission Year', 'Admission Month',
                                  'Admission Day', 'Admission Day of Week']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Gender', 'Blood Type', 'Medical Condition',
                        

In [9]:
# Fit on X_train ONLY, then transform X_train and X_test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Extract generated feature names from the OneHotEncoder
encoded_cat_features = list(preprocessor.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(CATEGORICAL_FEATURES))
all_processed_features = NUMERICAL_FEATURES + encoded_cat_features

print("--- PREPROCESSED DATA AUDIT ---")
print(f"X_train_processed shape: {X_train_processed.shape} (Rows, Encoded Features)")
print(f"X_test_processed shape:  {X_test_processed.shape}")
print(f"Total encoded features:  {len(all_processed_features)}")
print(f"\nAll Processed Feature Names ({len(all_processed_features)}):")
for i, name in enumerate(all_processed_features, 1):
    print(f"  {i:02d}. {name}")


--- PREPROCESSED DATA AUDIT ---
X_train_processed shape: (43972, 29) (Rows, Encoded Features)
X_test_processed shape:  (10994, 29)
Total encoded features:  29

All Processed Feature Names (29):
  01. Age
  02. Admission Year
  03. Admission Month
  04. Admission Day
  05. Admission Day of Week
  06. Gender_Female
  07. Gender_Male
  08. Blood Type_A+
  09. Blood Type_A-
  10. Blood Type_AB+
  11. Blood Type_AB-
  12. Blood Type_B+
  13. Blood Type_B-
  14. Blood Type_O+
  15. Blood Type_O-
  16. Medical Condition_Arthritis
  17. Medical Condition_Asthma
  18. Medical Condition_Cancer
  19. Medical Condition_Diabetes
  20. Medical Condition_Hypertension
  21. Medical Condition_Obesity
  22. Insurance Provider_Aetna
  23. Insurance Provider_Blue Cross
  24. Insurance Provider_Cigna
  25. Insurance Provider_Medicare
  26. Insurance Provider_UnitedHealthcare
  27. Admission Type_Elective
  28. Admission Type_Emergency
  29. Admission Type_Urgent


In [10]:
import joblib

# Create directory if it doesn't exist
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save preprocessor object and split arrays for subsequent modeling notebooks
joblib.dump(preprocessor, "../models/preprocessor.joblib")
joblib.dump({
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'X_train_processed': X_train_processed,
    'X_test_processed': X_test_processed,
    'feature_names': all_processed_features,
    'numerical_features': NUMERICAL_FEATURES,
    'categorical_features': CATEGORICAL_FEATURES
}, "../data/processed/split_data.joblib")

print("--- PIPELINE ARTIFACTS EXPORTED ---")
print("1. Saved preprocessor object -> ml/models/preprocessor.joblib")
print("2. Saved split datasets -> ml/data/processed/split_data.joblib")


--- PIPELINE ARTIFACTS EXPORTED ---
1. Saved preprocessor object -> ml/models/preprocessor.joblib
2. Saved split datasets -> ml/data/processed/split_data.joblib
